# ML_G0_P0002_answer

## 0. 정답본 범위
- Gate: G0
- Phase: P0002
- Topic: DataFrame vs ndarray/tensor
- Problem notebook: ML_G0_P0002.ipynb
- Correct answers included: Yes
- Output language: Korean
- Data directory: data/ML_G0_P0002

## 1. 핵심 기준표

| 구조 | 핵심 역할 | 주의할 점 |
|---|---|---|
| DataFrame | 사람이 읽는 표, column/index/dtype/결측치/EDA, feature 의미 확인 | 모델이 column 의미를 자동 이해하는 것은 아니다. |
| Series | 1차원 labeled array, DataFrame의 한 column 또는 y로 자주 사용 | `(n,)`과 `(n, 1)` 차이를 구분해야 한다. |
| ndarray | NumPy 수치 배열, column 이름 없음, 빠른 수치 연산, scikit-learn 입력에 자주 사용 | 변환 후 column 이름/label mapping이 사라질 수 있다. |
| tensor | 딥러닝 프레임워크의 다차원 수치 구조, batch dimension, GPU/자동미분과 연결 | ndarray와 비슷하지만 학습 그래프, device, gradient와 연결된다. |

## 2. 데이터 스키마

### students_scores.csv
- shape: `(8, 6)`
- 회귀 target 예시: `final_score`
- feature 후보: `study_hours`, `attendance_rate`, `assignment_score`, `midterm_score`
- 식별자 `student_id`는 보통 예측 feature에서 제외한다.

### customer_purchase.csv
- shape: `(8, 8)`
- 이진분류 target 예시: `purchased`
- categorical/object column: `city`, `device_type`
- 모델 입력 전 encoding 필요

### traffic_congestion.csv
- shape: `(8, 8)`
- 순서형 target: `congestion_level`
- label mapping: `0=원활`, `1=보통`, `2=혼잡`, `3=매우 혼잡`
- categorical column: `weekday`, `weather`

### mock_images.npz
- `images_gray.shape = (12, 28, 28)`
- `images_flat.shape = (12, 784)`
- `labels.shape = (12,)`


In [ ]:
# 정답본 공통 setup / 데이터 검증
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("data/ML_G0_P0002")
assert DATA_DIR.exists(), f"데이터 디렉터리가 없습니다: {DATA_DIR}"

expected_files = [
    "students_scores.csv",
    "customer_purchase.csv",
    "traffic_congestion.csv",
    "mock_images.npz",
]
for name in expected_files:
    assert (DATA_DIR / name).exists(), f"누락 파일: {name}"

students = pd.read_csv(DATA_DIR / "students_scores.csv")
customers = pd.read_csv(DATA_DIR / "customer_purchase.csv")
traffic = pd.read_csv(DATA_DIR / "traffic_congestion.csv")
images = np.load(DATA_DIR / "mock_images.npz")

assert students.shape == (8, 6)
assert customers.shape == (8, 8)
assert traffic.shape == (8, 8)
assert images["images_gray"].shape == (12, 28, 28)
assert images["images_flat"].shape == (12, 784)
assert images["labels"].shape == (12,)

print("데이터 검증 통과")
print("students:", students.shape)
print("customers:", customers.shape)
print("traffic:", traffic.shape)
print("images_gray:", images["images_gray"].shape)
print("images_flat:", images["images_flat"].shape)
print("labels:", images["labels"].shape)


## A-1. 네 가지 데이터 표현 비교

### 정답
DataFrame은 column 이름, index, dtype, 결측치 정보를 함께 가진 표 구조다. 사람이 데이터를 읽고 의미를 확인하며 EDA와 전처리 설계를 하기 좋다. Series는 DataFrame의 한 column처럼 쓰이는 1차원 labeled array이며, target `y`로 자주 사용된다. ndarray는 NumPy의 수치 배열이며 column 이름 없이 빠른 수치 계산에 적합하다. tensor는 딥러닝 프레임워크에서 쓰이는 다차원 수치 구조로, batch 연산, GPU, 자동미분과 연결된다.

### 근거
DataFrame/Series는 label과 의미 정보를 보존하고, ndarray/tensor는 계산 효율과 모델 입력에 초점이 있다.

### 자주 하는 오답
DataFrame을 모델이 column 의미까지 이해한다고 착각한다. ndarray에 column 이름이 남는다고 착각한다. tensor를 ndarray와 완전히 같은 것으로 설명한다.

### 최종 답안형
DataFrame은 의미 확인과 EDA를 위한 표, Series는 한 column 또는 1차원 y, ndarray는 이름 없는 수치 배열, tensor는 딥러닝 학습용 다차원 수치 구조다.


In [ ]:
# 정답 코드 예시 A-1: 네 구조의 실제 타입 비교
students = pd.read_csv(DATA_DIR / "students_scores.csv")

sample_df = students[["study_hours", "attendance_rate", "assignment_score", "midterm_score"]]
sample_series = students["final_score"]
sample_ndarray = sample_df.to_numpy()
# 실제 딥러닝 프레임워크 tensor는 사용하지 않고, 여기서는 tensor에 들어갈 수 있는 n차원 수치 배열 예시를 확인한다.
sample_tensor_like = images["images_gray"]

print("DataFrame:", type(sample_df), sample_df.shape, list(sample_df.columns))
print("Series:", type(sample_series), sample_series.shape, sample_series.name)
print("ndarray:", type(sample_ndarray), sample_ndarray.shape)
print("tensor-like image array:", type(sample_tensor_like), sample_tensor_like.shape)


## A-2. students_scores.csv에서 DataFrame과 ndarray 비교

### 정답
DataFrame 단계에서는 column 의미, dtype, 결측치, 값 범위, target 후보를 확인한다. `final_score`를 예측한다면 y는 `final_score`이고, X는 보통 `study_hours`, `attendance_rate`, `assignment_score`, `midterm_score` 같은 예측 변수다. `student_id`는 식별자라 일반 feature로 쓰기 부적절할 수 있다. X/y를 먼저 DataFrame/Series로 분리하면 어떤 값으로 무엇을 예측하는지 명확해진다. ndarray로 바꾸면 column 이름과 index, 의미 설명이 사라지고 모델은 숫자 행렬만 계산한다.

### 근거
DataFrame에서 의미를 확인한 뒤 X/y를 나누고, 모델 계산 단계에서 ndarray로 바꾼다.

### 자주 하는 오답
`student_id`를 무조건 feature에 넣는다. `final_score`를 X에 포함해 target leakage를 만든다. `to_numpy()` 후에도 column 이름이 남는다고 생각한다.

### 최종 답안형
`final_score`는 y이고, 나머지 학습 관련 점수/출석/공부시간은 X 후보다. ndarray 변환 후 모델은 column 이름이 아니라 숫자 배열만 본다.


In [ ]:
# 정답 코드 예시 A-2: students_scores.csv의 X/y 분리와 ndarray 변환
students = pd.read_csv(DATA_DIR / "students_scores.csv")

feature_cols = ["study_hours", "attendance_rate", "assignment_score", "midterm_score"]
target_col = "final_score"

X_df = students[feature_cols]
y = students[target_col]
X_np = X_df.to_numpy()
y_np = y.to_numpy()

print("X_df columns:", list(X_df.columns))
print("X_df shape:", X_df.shape)
print("y type/shape:", type(y).__name__, y.shape)
print("X_np shape:", X_np.shape)
print("y_np shape:", y_np.shape)
print("ndarray에는 column 이름이 없다:", hasattr(X_np, "columns"))


## B-1. `head()`, `info()`, `describe()`의 역할

### 정답
`head()`는 앞부분 샘플을 보며 column 이름, 값 예시, 데이터가 예상대로 읽혔는지 확인한다. `info()`는 column별 non-null count와 dtype을 확인해 결측치와 object/string column을 찾는다. `describe()`는 수치형 column의 count, mean, std, min/max, quartile을 확인한다. 모델에 넣기 전 `info()`가 중요한 이유는 object dtype이나 결측치가 있으면 바로 수치 모델에 넣기 어렵기 때문이다. `describe()`만으로는 문자열 category, label mapping, column 의미, 결측치의 맥락을 충분히 알 수 없다.

### 근거
EDA 함수는 모델링 전 데이터 구조와 위험 요소를 확인하기 위한 도구다.

### 자주 하는 오답
`describe()`만 보면 전체 데이터 이해가 끝난다고 생각한다. `info()`의 dtype 확인을 무시한다.

### 최종 답안형
`head()`는 샘플, `info()`는 dtype/결측치, `describe()`는 수치 통계를 확인한다. 특히 object dtype은 모델 입력 전 encoding이 필요하다.


In [ ]:
# 정답 코드 예시 B-1: head/info/describe 확인
customers = pd.read_csv(DATA_DIR / "customer_purchase.csv")

print("head()")
display(customers.head())

print("info()")
customers.info()

print("describe()")
display(customers.describe())

print("object/string dtype columns:", customers.select_dtypes(include=["object", "string"]).columns.tolist())


## B-2. Series와 DataFrame 한 열의 shape 차이

### 정답
`y_series = students["final_score"]`는 `pandas.Series`이고 보통 shape는 `(8,)`이다. `y_df = students[["final_score"]]`는 한 열짜리 `pandas.DataFrame`이고 shape는 `(8, 1)`이다. 둘 다 같은 값을 담을 수 있지만 차원이 다르다.

### 근거
Series는 1차원 labeled array이고, DataFrame은 2차원 labeled table이다. scikit-learn의 target `y`는 보통 Series처럼 `(n,)` 형태를 자주 기대한다. 반면 DataFrame 한 열은 `(n, 1)`이라 모델이나 함수에 따라 warning이나 shape 처리가 달라질 수 있다.

### 자주 하는 오답
`students["final_score"]`와 `students[["final_score"]]`가 완전히 같은 구조라고 생각한다. Series를 행렬이라고 설명한다. `(8,)`과 `(8, 1)`의 차이를 무시한다.

### 최종 답안형
Series 한 열 선택은 1차원 `y`인 `(8,)`이고, DataFrame 한 열 선택은 2차원 table인 `(8, 1)`이다. 값은 비슷해도 자료구조와 shape가 다르므로 모델 입력 전 확인해야 한다.


In [ ]:
# 정답 코드 예시 B-2: Series vs 한 열짜리 DataFrame
students = pd.read_csv(DATA_DIR / "students_scores.csv")

y_series = students["final_score"]
y_df = students[["final_score"]]

print("y_series type:", type(y_series).__name__, "shape:", y_series.shape)
print("y_df type:", type(y_df).__name__, "shape:", y_df.shape)
print("y_series ndim:", y_series.ndim)
print("y_df ndim:", y_df.ndim)


## C-1. 식별자 column과 feature column 구분

### 정답
`final_score`를 예측하는 문제에서 target은 `final_score`다. feature로는 `study_hours`, `attendance_rate`, `assignment_score`, `midterm_score`가 자연스럽다. `student_id`는 식별자이므로 기본 feature에서 제외한다.

### 근거
`student_id`는 학생을 구분하기 위한 번호이지 기말 점수를 설명하는 실제 학습 변수라고 보기 어렵다. ID를 feature로 넣으면 모델이 번호의 크기나 순서를 의미 있는 수치로 오해할 수 있다. 또한 target인 `final_score`가 X에 들어가면 target leakage가 생긴다.

### 자주 하는 오답
`student_id`를 숫자 column이므로 무조건 feature에 넣는다. `final_score`를 X에도 넣고 y에도 넣는다. column이 숫자면 모두 모델 입력으로 안전하다고 생각한다.

### 최종 답안형
X는 `study_hours`, `attendance_rate`, `assignment_score`, `midterm_score`이고, y는 `final_score`다. `student_id`는 식별자라 제외하고, target column은 X에 넣지 않는다.


In [ ]:
# 정답 코드 예시 C-1: 식별자와 target 제외
students = pd.read_csv(DATA_DIR / "students_scores.csv")

feature_cols = ["study_hours", "attendance_rate", "assignment_score", "midterm_score"]
target_col = "final_score"
excluded_cols = ["student_id"]

X_df = students[feature_cols]
y = students[target_col]

print("excluded_cols:", excluded_cols)
print("feature_cols:", feature_cols)
print("target_col:", target_col)
print("X_df shape:", X_df.shape)
print("y shape:", y.shape)
assert target_col not in feature_cols
assert "student_id" not in feature_cols


## C-2. categorical column을 가진 DataFrame의 모델 입력 준비

### 정답
`customer_purchase.csv`에서 target은 `purchased`다. 숫자 feature 후보는 `age`, `income`, `visit_count`, `cart_amount`이고, categorical feature 후보는 `city`, `device_type`이다. `customer_id`는 식별자라 일반 feature에서 제외하는 것이 안전하다.

### 근거
`city`, `device_type`은 object/string dtype이므로 숫자 모델에 그대로 넣기 어렵다. 바로 `to_numpy()`를 하면 문자열이 섞인 object 배열이 되거나 모델 입력에서 오류가 날 수 있다. 모델 입력 전 one-hot encoding 같은 categorical encoding이 필요하다. 이 판단은 `customers.info()`로 dtype을 확인해야 가능하다.

### 자주 하는 오답
문자열 column도 `to_numpy()`만 하면 자동으로 좋은 숫자 feature가 된다고 생각한다. `purchased`를 feature에 포함한다. `customer_id`를 의미 있는 수치 feature처럼 사용한다.

### 최종 답안형
`purchased`는 y이고, 숫자 feature와 categorical feature를 분리해 확인한다. `city`, `device_type`은 encoding 전 모델 입력에 바로 넣지 않으며, `info()`로 dtype을 먼저 점검한다.


In [ ]:
# 정답 코드 예시 C-2: categorical column 확인과 encoding 방향
customers = pd.read_csv(DATA_DIR / "customer_purchase.csv")

target_col = "purchased"
id_cols = ["customer_id"]
numeric_feature_cols = ["age", "income", "visit_count", "cart_amount"]
categorical_feature_cols = ["city", "device_type"]

print("dtypes:")
print(customers.dtypes)
print("numeric_feature_cols:", numeric_feature_cols)
print("categorical_feature_cols:", categorical_feature_cols)
print("target_col:", target_col)

X_encoded_example = pd.get_dummies(customers[numeric_feature_cols + categorical_feature_cols], columns=categorical_feature_cols)
y = customers[target_col]
print("encoded columns:", X_encoded_example.columns.tolist())
print("X_encoded_example shape:", X_encoded_example.shape)
print("y shape:", y.shape)


## C-3. DataFrame에서 ndarray로 변환하기 전 기록해야 할 것

### 정답
변환 전에는 `feature_cols` 목록과 순서, target column, categorical encoding 방식, label mapping, dtype/결측치 처리 방식을 기록해야 한다. ndarray에는 값과 shape는 남지만 column 이름, index, feature 의미, label 이름은 자동으로 남지 않는다.

### 근거
모델은 ndarray의 첫 번째 열이 어떤 feature인지 알지 못하고, 지정된 순서의 숫자 배열로만 계산한다. 학습 때와 예측 때 column 순서가 바뀌면 완전히 다른 의미의 값을 같은 feature 위치에 넣는 문제가 생긴다. `congestion_level`처럼 숫자 label이 class 의미를 가진 경우 `0=원활` 같은 mapping도 별도로 보존해야 한다.

### 자주 하는 오답
ndarray에 column 이름이 보존된다고 생각한다. feature_cols 순서를 기록하지 않는다. label mapping이 모델 객체나 ndarray에 자동 저장된다고 생각한다.

### 최종 답안형
DataFrame에서 ndarray로 바꾸기 전 `feature_cols` 순서, target, encoding, label mapping을 기록해야 한다. ndarray에는 숫자 값과 shape만 남고 column 의미는 사라진다.


In [ ]:
# 정답 코드 예시 C-3: ndarray 변환 전 기록해야 하는 metadata
customers = pd.read_csv(DATA_DIR / "customer_purchase.csv")

feature_cols = ["age", "income", "visit_count", "cart_amount"]
target_col = "purchased"
X_df = customers[feature_cols]
y = customers[target_col]
X_np = X_df.to_numpy()
metadata = {
    "feature_cols": feature_cols,
    "target_col": target_col,
    "X_shape": X_np.shape,
    "y_shape": y.shape,
}

print("metadata:", metadata)
print("X_np shape:", X_np.shape)
print("first row values:", X_np[0].tolist())
print("첫 번째 값의 의미는 metadata의 feature_cols[0]로만 알 수 있음:", metadata["feature_cols"][0])


## D-1. ndarray와 tensor의 역할 차이

### 정답
ndarray는 NumPy의 다차원 수치 배열로 빠른 수치 계산과 scikit-learn 입력에 자주 사용된다. tensor는 딥러닝 프레임워크에서 사용하는 다차원 수치 구조로, GPU 연산, 자동미분, batch 학습과 연결된다. 둘 다 숫자를 축과 shape를 가진 배열로 다룬다는 공통점이 있다. DataFrame과 가장 다른 점은 column 이름, dtype 요약, index 같은 의미 정보를 중심에 두지 않고 계산 가능한 수치 구조를 중심에 둔다는 점이다.

### 근거
tensor는 단순 배열 이상의 학습 프레임워크 객체다.

### 자주 하는 오답
tensor를 그냥 DataFrame의 다른 이름으로 생각한다. tensor와 ndarray의 차이를 전혀 설명하지 않는다.

### 최종 답안형
ndarray는 NumPy 수치 배열, tensor는 딥러닝 학습과 자동미분/GPU에 연결되는 다차원 수치 구조다.


In [ ]:
# 정답 코드 예시 D-1: ndarray와 tensor-like n차원 배열의 shape 비교
students = pd.read_csv(DATA_DIR / "students_scores.csv")
X_np = students[["study_hours", "attendance_rate", "assignment_score", "midterm_score"]].to_numpy()
images_gray = images["images_gray"]

print("tabular ndarray shape:", X_np.shape)
print("image tensor-like shape:", images_gray.shape)
print("tabular ndim:", X_np.ndim)
print("image ndim:", images_gray.ndim)


## D-2. mock_images.npz와 이미지 tensor 표현

### 정답
`(12, 28, 28)`은 12장의 28x28 grayscale 이미지 묶음이다. 첫 번째 축 12는 batch/sample 개수이고, 뒤의 두 축은 이미지의 세로와 가로 공간 구조다. `(12, 784)`는 각 28x28 이미지를 784개 값의 1차원 feature vector로 펼친 것이다. `labels.shape = (12,)`는 12개 이미지 각각에 대응하는 1차원 class label이다. 이미지는 pixel의 위치와 이웃 관계가 중요하므로 DataFrame보다 ndarray/tensor가 자연스럽다. Flatten하면 2D 공간 구조와 인접 pixel 관계가 사라진다.

### 근거
이미지 데이터는 batch와 spatial dimension을 가진다.

### 자주 하는 오답
`(12, 28, 28)`을 12행 28열 표로 오해한다. Flatten 후에도 공간 정보가 그대로 있다고 생각한다.

### 최종 답안형
`images_gray`는 12장의 28x28 이미지, `images_flat`은 이를 784차원 vector로 펼친 배열, `labels`는 각 이미지의 class label이다.


In [ ]:
# 정답 코드 예시 D-2: mock_images.npz shape 확인
images_gray = images["images_gray"]
images_flat = images["images_flat"]
labels = images["labels"]

print("images_gray shape:", images_gray.shape)
print("images_flat shape:", images_flat.shape)
print("labels shape:", labels.shape)
print("flatten 검증:", np.array_equal(images_gray.reshape(12, 784), images_flat))
print("labels:", labels.tolist())


## E-1. `model.fit(X_train, y_train)` 해석

### 정답
`X_train`은 학습용 feature matrix이고 `y_train`은 각 sample의 target이다. `model.fit(X_train, y_train)`은 X의 패턴을 보고 y를 맞히도록 모델 파라미터를 학습하는 과정이다. 일부 라이브러리는 DataFrame을 입력으로 받을 수 있지만, 내부 계산은 결국 수치 배열 기반으로 진행된다. 모델이 실제로 보는 것은 column 이름의 의미라기보다 shape와 dtype이 맞는 숫자 값들이다.

### 근거
fit은 X와 y의 대응 관계를 학습한다.

### 자주 하는 오답
`fit`이 데이터를 그냥 저장한다고 생각한다. y 없이도 지도학습 fit이 된다고 착각한다.

### 최종 답안형
`X_train`은 입력 feature, `y_train`은 target이며, `fit`은 X로 y를 예측하도록 모델을 학습시키는 과정이다.


In [ ]:
# 정답 코드 예시 E-1: model.fit에 들어갈 X/y 구조 예시
students = pd.read_csv(DATA_DIR / "students_scores.csv")
X_train = students[["study_hours", "attendance_rate", "assignment_score", "midterm_score"]].to_numpy()
y_train = students["final_score"].to_numpy()

print("model.fit(X_train, y_train)에 들어갈 수 있는 구조 예시")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("sample count 일치:", X_train.shape[0] == y_train.shape[0])


## E-2. scikit-learn 입력과 Keras 입력

### 정답
`clf.fit(X_train_np, y_train)`은 scikit-learn 모델에 ndarray 또는 DataFrame 형태의 feature matrix와 target을 넣는 구조다. `model.fit(X_train_tensor, y_train_tensor)`는 Keras/PyTorch 계열에서 tensor 또는 tensor로 변환 가능한 배열을 넣어 학습하는 구조다. 공통점은 X와 y의 sample 개수가 맞아야 하고 수치 계산 가능한 구조여야 한다는 점이다. 차이점은 딥러닝 tensor는 GPU, 자동미분, batch 연산과 더 직접적으로 연결된다는 점이다. DataFrame 단계는 여전히 column 의미 확인, dtype 점검, encoding 설계, leakage 방지에 필요하다.

### 근거
모델 입력 직전 구조가 달라도 DataFrame 단계의 의미 검사는 사라지지 않는다.

### 자주 하는 오답
Keras는 DataFrame 의미를 그대로 이해한다고 생각한다. scikit-learn과 Keras의 입력 차이를 프레임워크 이름만으로 설명한다.

### 최종 답안형
scikit-learn은 ndarray/DataFrame 입력을 자주 받고, Keras/PyTorch는 tensor 중심이지만, 둘 다 수치 구조의 X/y를 학습한다.


In [ ]:
# 정답 코드 예시 E-2: scikit-learn 입력과 Keras 입력의 shape 관점 비교
students = pd.read_csv(DATA_DIR / "students_scores.csv")
X_train_np = students[["study_hours", "attendance_rate", "assignment_score", "midterm_score"]].to_numpy()
y_train = students["final_score"].to_numpy()
X_train_tensor_like = X_train_np.astype("float32")
y_train_tensor_like = y_train.astype("float32")

print("clf.fit에 자주 들어가는 ndarray:", X_train_np.shape, X_train_np.dtype)
print("model.fit에 들어갈 수 있는 tensor-like array:", X_train_tensor_like.shape, X_train_tensor_like.dtype)
print("target:", y_train_tensor_like.shape, y_train_tensor_like.dtype)


## F-1. object dtype 함정

### 정답
문제가 되는 column은 `city`, `device_type`이다. 이들은 문자열/object dtype이므로 숫자 배열 기반 모델에 그대로 넣기 어렵다. 모델에 넣기 전 one-hot encoding, ordinal encoding, embedding 등 적절한 categorical encoding이 필요하다. DataFrame 단계에서는 `customers.info()`로 dtype을 먼저 확인해야 한다.

### 근거
문자열 category는 수치 모델 입력 전 처리 대상이다.

### 자주 하는 오답
`to_numpy()`만 하면 문자열도 자동으로 좋은 숫자 feature가 된다고 생각한다.

### 최종 답안형
`city`, `device_type`은 object column이므로 encoding 후 모델 입력에 사용해야 한다.


In [ ]:
# 정답 코드 예시 F-1: object dtype 함정 확인
customers = pd.read_csv(DATA_DIR / "customer_purchase.csv")

object_cols = customers.select_dtypes(include=["object", "string"]).columns.tolist()
print("object/string columns:", object_cols)
print(customers[object_cols].head())

raw_np = customers.drop(columns=["purchased"]).to_numpy()
print("raw_np dtype:", raw_np.dtype)
print("문자열이 섞이면 수치 모델 입력 전 encoding 필요")


## F-2. column 의미 상실 함정

### 정답
배열만 보고는 각 열이 age인지 income인지 visit_count인지 확정할 수 없다. DataFrame이었다면 column name이 있어 각 열의 의미를 알 수 있다. ndarray로 변환하기 전 `feature_cols`와 column 순서를 기록해야 한다. column 순서가 바뀌면 모델은 같은 위치의 값을 같은 feature로 해석하므로, age 자리에 income이 들어가는 식의 치명적 입력 오류가 생길 수 있다.

### 근거
ndarray는 숫자 위치만 보존하고 column 의미는 보존하지 않는다.

### 자주 하는 오답
배열 값의 크기만 보고 column 의미를 추측할 수 있다고 생각한다.

### 최종 답안형
ndarray 변환 전 feature column 목록과 순서를 반드시 보존해야 한다.


In [ ]:
# 정답 코드 예시 F-2: ndarray column 의미 상실
arr = np.array([[23, 3500, 5], [41, 5200, 2], [35, 4300, 8]])
feature_cols = ["age", "income", "visit_count"]

print("arr shape:", arr.shape)
print("arr만 보면 column 이름은 없음")
print("feature_cols를 따로 기록해야 함:", feature_cols)
print("첫 번째 열 의미:", feature_cols[0], "=", arr[:, 0].tolist())


## G-1. traffic_congestion.csv의 target 의미 보존

### 정답
`congestion_level`은 교통 혼잡도 target이다. `0,1,2,3`은 실제 수치량이라기보다 `원활/보통/혼잡/매우 혼잡` class를 나타내는 순서형 label이다. DataFrame에는 column 이름 `congestion_level`과 주변 문서/label mapping을 통해 의미가 보존된다. ndarray로 바꾸면 값 `0,1,2,3`만 남고 이것이 무엇을 뜻하는지 자동으로 남지 않는다. 예측 결과를 해석하려면 label mapping을 별도 dict나 문서로 보존해야 한다.

### 근거
숫자 label은 ndarray 변환 후 의미 설명이 사라지기 쉽다.

### 자주 하는 오답
`0,1,2,3`이 ndarray 안에 있으니 class 이름도 자동 보존된다고 생각한다.

### 최종 답안형
`congestion_level`은 순서형 class target이며, ndarray 변환 후에도 `0=원활`, `1=보통`, `2=혼잡`, `3=매우 혼잡` mapping을 따로 보존해야 한다.


In [ ]:
# 정답 코드 예시 G-1: target label mapping 보존
traffic = pd.read_csv(DATA_DIR / "traffic_congestion.csv")
label_mapping = {0: "원활", 1: "보통", 2: "혼잡", 3: "매우 혼잡"}

X_df = traffic.drop(columns=["congestion_level"])
y = traffic["congestion_level"]

print("target unique values:", sorted(y.unique()))
print("label_mapping:", label_mapping)
print("y as names:", y.map(label_mapping).tolist())
print("X_df columns before ndarray:", X_df.columns.tolist())


## G-2. 최종 한 문장 요약

### 정답
DataFrame은 사람이 column 의미, dtype, 결측치, category, target 후보를 확인하고 전처리 전략을 세우기 위해 필요하다. ndarray/tensor는 모델이 빠르게 수치 계산을 하고 학습하기 위해 필요하다. 둘 사이를 변환할 때는 column 이름, column 순서, dtype, 결측치, categorical encoding, label mapping, target leakage를 조심해야 한다.

### 근거
의미 확인 단계와 계산 단계의 역할을 분리한다.

### 자주 하는 오답
DataFrame은 초보자용이고 모델에는 항상 불필요하다고 생각한다. 변환 후 의미 정보 손실을 무시한다.

### 최종 답안형
같은 데이터라도 DataFrame으로는 의미와 구조를 확인하고, ndarray/tensor로는 모델이 계산할 수 있는 수치 구조로 학습한다.


In [ ]:
# 정답 코드 예시 G-2: 전체 흐름 요약용 객체 확인
students = pd.read_csv(DATA_DIR / "students_scores.csv")
feature_cols = ["study_hours", "attendance_rate", "assignment_score", "midterm_score"]
target_col = "final_score"

X_df = students[feature_cols]
y_series = students[target_col]
X_np = X_df.to_numpy()
y_np = y_series.to_numpy()

print("1. DataFrame 단계: 의미 확인", type(X_df).__name__, X_df.shape, list(X_df.columns))
print("2. Series target:", type(y_series).__name__, y_series.shape, y_series.name)
print("3. ndarray 계산 단계:", type(X_np).__name__, X_np.shape)
print("4. 변환 전 feature_cols 기록:", feature_cols)


## 3. 채점 기준

| 항목 | 배점 관점 |
|---|---|
| DataFrame/Series/ndarray/tensor 역할 구분 | 각 구조의 목적과 정보 보존 범위를 구분하는가 |
| X/y와 shape 설명 | feature matrix와 target vector를 구분하고 shape를 설명하는가 |
| dtype/categorical 처리 | object/string column을 모델 입력 전 처리해야 함을 아는가 |
| 변환 시 정보 손실 | column name, label mapping, spatial structure 손실을 설명하는가 |
| 모델 입력 관점 | `model.fit(X_train, y_train)`의 의미를 구조적으로 해석하는가 |

## 4. 오답튜터 기준표

1. DataFrame을 모델이 의미까지 이해한다고 착각
2. ndarray에 column 이름이 보존된다고 착각
3. object/string dtype을 그대로 모델에 넣으려 함
4. Series와 DataFrame의 shape 차이를 놓침
5. tensor를 단순히 ndarray와 완전히 같은 것으로 설명
6. 이미지 데이터를 표 형태로만 생각하고 batch/spatial 구조를 놓침
7. label mapping이 ndarray 변환 후에도 자동 보존된다고 착각

## 5. 다음 Phase 연결

다음 Phase에서는 실제 전처리 흐름으로 이어진다.

- categorical encoding
- train/test split
- scaling
- leakage 방지
- 모델 입력 전 최종 X/y 검증
